# 11｜YOLOv1 的坐标换算：从一个网格回到原图

上一课，我们已经看懂了 YOLOv1 的预测张量：

$$
S\times S\times(5B+C)
$$

其中，每一个候选框都要预测四个数字 $x,y,w,h$。

这一课只解决一个问题：**这四个数字究竟怎样变成原图上的矩形框？**

整节课只跟踪一个框，不讨论损失函数、IoU 和 NMS。


## 一、先记住一句最重要的话

在 YOLOv1 中：

- $x,y$ 描述框中心在**负责网格内部**的相对位置；
- $w,h$ 描述框宽高占**整张图片**宽高的比例。

也就是说，$x,y$ 和 $w,h$ 的参照物不一样。

这正是坐标换算最容易混淆的地方。


## 二、为什么不直接预测像素坐标

假设同一只狗：

- 在 $448\times448$ 图片里，中心可能是 $(208,166.4)$；
- 图片缩放以后，像素坐标会跟着改变；
- 但“它位于某个网格内部约四分之一的位置”比较稳定。

因此，网络先预测相对坐标，再把相对坐标换回像素坐标。

可以先把整个过程记成：

$$
\text{网格编号}+\text{格内偏移}
\longrightarrow
\text{全图归一化坐标}
\longrightarrow
\text{像素坐标}
$$


## 三、这一课会遇到三套坐标

为了不把数字混在一起，我们先给三套坐标起名字。

| 坐标系统 | 例子 | 它在回答什么 |
|---|---:|---|
| 网格编号 | $(c_x,c_y)=(3,2)$ | 框中心落在第几个网格 |
| 网络输出 | $(x,y,w,h)$ | 中心在格内哪里，框占全图多大 |
| 原图像素 | $(X_c,Y_c,W_{box},H_{box})$ | 框中心和宽高各是多少像素 |

这里：

- $c_x$ 表示列编号，从左向右数；
- $c_y$ 表示行编号，从上向下数；
- 编号都从 $0$ 开始。

所以 $(3,2)$ 是人眼所说的“第 4 列、第 3 行”。


## 四、网格编号和 $x,y$ 各管一部分

假设图片被分成 $S\times S$ 个网格。

只知道 $x=0.25$，我们还不知道框中心在整张图的哪里。因为每一个网格内部都可以有一个“横向四分之一处”。

必须把两部分合在一起：

- $c_x,c_y$ 告诉我们先走到哪个网格；
- $x,y$ 告诉我们再在这个网格内部走多远。

可以把 $c_x,c_y$ 想成“街区编号”，把 $x,y$ 想成“进入街区后的位置”。


## 五、先从横坐标推一遍

一行共有 $S$ 个格子。框中心位于第 $c_x$ 列，并且在这个格子内部走了 $x$ 个格宽。

因此，从整张图左边缘到框中心，一共走了：

$$
c_x+x
$$

个格宽。

整张图共有 $S$ 个格宽，所以框中心相对整张图的横坐标是：

$$
x_g=\frac{c_x+x}{S}
$$

$x_g$ 的下标 $g$ 表示 global，也就是“全图坐标”。


## 六、纵坐标完全同理

框中心位于第 $c_y$ 行，并且在这个格子内部向下走了 $y$ 个格高。

因此：

$$
y_g=\frac{c_y+y}{S}
$$

合在一起就是：

$$
\boxed{
x_g=\frac{c_x+x}{S},\qquad
y_g=\frac{c_y+y}{S}
}
$$

这一小步的含义是：把“哪个格子 + 格内位置”合成为“整张图中的相对位置”。


## 七、$w,h$ 不需要再除以网格数

YOLOv1 中，$w,h$ 本来就是相对于整张图片的比例：

$$
w_g=w,\qquad h_g=h
$$

例如：

- $w=0.40$ 表示框宽是图片宽度的 $40\%$；
- $h=0.30$ 表示框高是图片高度的 $30\%$。

它们不是“占一个网格的 $40\%$ 和 $30\%$”，所以不能再除以 $S$。


## 八、从全图比例变成像素

设原图宽为 $W_{img}$，高为 $H_{img}$。

框中心的像素坐标为：

$$
X_c=x_gW_{img},\qquad Y_c=y_gH_{img}
$$

框的像素宽高为：

$$
W_{box}=wW_{img},\qquad H_{box}=hH_{img}
$$

到这里，我们得到的是中心点加宽高，也就是：

$$
(X_c,Y_c,W_{box},H_{box})
$$


## 九、完整算例：只跟踪一个框

现在用一组具体数字慢慢计算。

已知：

- 图片大小为 $448\times448$；
- YOLOv1 把图片分成 $7\times7$ 个网格，即 $S=7$；
- 框中心落在 $(c_x,c_y)=(3,2)$ 这个格子；
- 网络为这个候选框输出 $(x,y,w,h)=(0.25,0.60,0.40,0.30)$。

因为编号从 $0$ 开始，$(3,2)$ 就是第 4 列、第 3 行。


## 十、先计算每个网格有多大

图片宽高都是 $448$，每一行和每一列都有 $7$ 个网格：

$$
\frac{448}{7}=64
$$

所以每个网格的大小是：

$$
64\times64\ \text{像素}
$$

这个数字可以帮助我们用肉眼检查后面的计算是否合理。


## 十一、把中心点换成全图比例

横坐标：

$$
x_g=\frac{3+0.25}{7}=\frac{3.25}{7}\approx0.4643
$$

纵坐标：

$$
y_g=\frac{2+0.60}{7}=\frac{2.60}{7}\approx0.3714
$$

这表示框中心大约位于整张图宽度的 $46.43\%$、高度的 $37.14\%$ 处。


## 十二、把中心点换成像素

$$
X_c=0.4643\times448=208
$$

$$
Y_c=0.3714\times448=166.4
$$

因此，框中心约为：

$$
(X_c,Y_c)=(208,166.4)
$$

也可以直接从网格出发检查：

$$
X_c=3\times64+0.25\times64=208
$$

$$
Y_c=2\times64+0.60\times64=166.4
$$

两种算法得到相同结果。


## 十三、把宽高换成像素

因为 $w,h$ 相对于整张图：

$$
W_{box}=0.40\times448=179.2
$$

$$
H_{box}=0.30\times448=134.4
$$

所以中心点加宽高形式的框是：

$$
\boxed{(208,166.4,179.2,134.4)}
$$

注意，$179.2$ 明显大于一个网格的宽度 $64$。这完全正常。


## 十四、再换成左上角和右下角

很多程序最终需要 $xyxy$ 格式，也就是：

$$
(x_1,y_1,x_2,y_2)
$$

从中心点向左右各走半个框宽，向上下各走半个框高：

$$
\begin{aligned}
x_1&=X_c-\frac{W_{box}}{2}, & y_1&=Y_c-\frac{H_{box}}{2},\\
x_2&=X_c+\frac{W_{box}}{2}, & y_2&=Y_c+\frac{H_{box}}{2}.
\end{aligned}
$$

代入数字：

$$
\begin{aligned}
x_1&=208-89.6=118.4,\\
y_1&=166.4-67.2=99.2,\\
x_2&=208+89.6=297.6,\\
y_2&=166.4+67.2=233.6.
\end{aligned}
$$

最终像素框是：

$$
\boxed{(118.4,99.2,297.6,233.6)}
$$


## 十五、为什么框可以伸出负责它的网格

负责网格只规定一件事：**物体中心落在哪个网格，就由哪个网格负责这个物体。**

它没有规定整个物体必须待在这个网格里。

在刚才的例子中：

- 负责网格只有 $64\times64$ 像素；
- 预测框却有 $179.2\times134.4$ 像素；
- 框自然会跨过多个网格。

因此要分清：

$$
\text{网格决定责任位置，不是框的活动边界。}
$$


## 十六、训练时，真实框怎样变成监督答案

预测时，我们把 $(x,y,w,h)$ 解码成像素框。训练前还要做一次相反的换算：把标注框变成网络要学习的答案。

设标注框为 $(x_1,y_1,x_2,y_2)$，先求中心和宽高：

$$
X_c=\frac{x_1+x_2}{2},\qquad
Y_c=\frac{y_1+y_2}{2}
$$

$$
W_{box}=x_2-x_1,\qquad
H_{box}=y_2-y_1
$$

再除以图片宽高，得到全图归一化坐标：

$$
x_g=\frac{X_c}{W_{img}},\qquad
y_g=\frac{Y_c}{H_{img}},\qquad
w=\frac{W_{box}}{W_{img}},\qquad
h=\frac{H_{box}}{H_{img}}
$$


## 十七、怎样找到负责网格和格内偏移

得到全图中心 $(x_g,y_g)$ 后，负责网格编号为：

$$
c_x=\left\lfloor Sx_g\right\rfloor,\qquad
c_y=\left\lfloor Sy_g\right\rfloor
$$

去掉整数部分，剩下的就是格内偏移：

$$
x=Sx_g-c_x,\qquad y=Sy_g-c_y
$$

对刚才的框反向检查：

$$
7\times0.4643\approx3.25
$$

整数部分 $3$ 是列编号，剩下的 $0.25$ 是格内横向偏移。

同理，$7\times0.3714\approx2.60$，得到行编号 $2$ 和格内纵向偏移 $0.60$。


## 十八、把编码和解码连起来

训练数据准备阶段：

$$
\text{真实像素框}
\longrightarrow
\text{负责网格与 }(x,y,w,h)
$$

模型预测以后：

$$
\text{负责网格与 }(x,y,w,h)
\longrightarrow
\text{预测像素框}
$$

前者常叫**编码**，后者常叫**解码**。

模型并不是直接把四个像素坐标背下来，而是在学习一种约定好的坐标表达方式。


## 十九、边界情况：算出的框超出图片怎么办

网络刚开始训练时，预测可能很不准确，解码后的 $x_1,y_1,x_2,y_2$ 可能超出图片边界。

例如 $x_1=-12$，表示框的左边跑到了图片左侧之外。

在实际推理实现中，通常会把最终坐标裁剪到图片范围内：

$$
0\le x_1,x_2\le W_{img},\qquad
0\le y_1,y_2\le H_{img}
$$

“裁剪坐标”只是输出处理，不代表网络从来不会预测到图外。


## 二十、五个最常见的混淆

1. **把 $x,y$ 当成全图坐标。** YOLOv1 的 $x,y$ 是格内偏移，还要结合网格编号。
2. **把 $w,h$ 当成格内比例。** YOLOv1 的 $w,h$ 相对于整张图。
3. **把行列写反。** $c_x$ 是列，$c_y$ 是行。
4. **忘记编号从 $0$ 开始。** 第 4 列的编号是 $3$。
5. **把中心宽高格式和四角格式混用。** $(X_c,Y_c,W,H)$ 不是 $(x_1,y_1,x_2,y_2)$。

还要注意：后续 YOLO 版本会采用不同的参数化方法。本课公式只用来理解 YOLOv1，不要机械套到所有版本。


## 二十一、现在检查自己是否真的懂了

仍然使用 $448\times448$ 图片和 $7\times7$ 网格。

某个候选框来自网格 $(c_x,c_y)=(1,4)$，网络输出：

$$
(x,y,w,h)=(0.5,0.25,0.20,0.10)
$$

请先不看答案，回答：

1. 框中心的全图归一化坐标是多少？
2. 框中心的像素坐标是多少？
3. 框的像素宽高是多少？
4. 为什么这个框可以跨出 $(1,4)$ 网格？


## 二十二、自测答案

中心的全图归一化坐标：

$$
x_g=\frac{1+0.5}{7}=\frac{1.5}{7},\qquad
y_g=\frac{4+0.25}{7}=\frac{4.25}{7}
$$

中心的像素坐标：

$$
X_c=\frac{1.5}{7}\times448=96,\qquad
Y_c=\frac{4.25}{7}\times448=272
$$

框的像素宽高：

$$
W_{box}=0.20\times448=89.6,\qquad
H_{box}=0.10\times448=44.8
$$

它可以跨出负责网格，因为网格只负责框的中心，框的宽高相对于整张图片。


## 二十三、本课总结

只要牢牢记住参照物，坐标换算就不会乱：

$$
\boxed{
\begin{aligned}
x_g&=\frac{c_x+x}{S}, & y_g&=\frac{c_y+y}{S},\\
X_c&=x_gW_{img}, & Y_c&=y_gH_{img},\\
W_{box}&=wW_{img}, & H_{box}&=hH_{img}.
\end{aligned}
}
$$

一句话复述：

> $c_x,c_y$ 选择网格，$x,y$ 确定中心在格内的位置，$w,h$ 确定框占整张图多大。


## 下一课：YOLOv1 的预测方式与 IoU

下一课接着回答一个自然问题：

一个网格预测了两个候选框，但一个真实物体只需要一个主要负责的框，那么训练时到底选谁？

我们会继续只跟踪一个物体，理解：

- 为什么同一网格要准备两个候选框；
- IoU 在“选择负责框”时做了什么；
- 置信度到底同时包含了哪两层含义；
- “网格负责物体”和“候选框负责回归”为什么不是同一句话。


## 参考来源

- [薛什么的B站《零基础入门深度学习》课程目录](https://www.bilibili.com/cheese/play/ss794934934)：本课沿其“6.3 YOLOv1 坐标换算”的顺序展开，并补充完整数值算例。
- [YOLOv1 原论文：You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640)：$x,y$ 相对于网格边界，$w,h$ 相对于整张图片。
